# 🔄 Double Pendulum RHONN Identification

## System Description

This notebook implements **RHONN (Recurrent High Order Neural Network) identification** for a **Double Pendulum System** using both **EKF (Extended Kalman Filter)** and **Particle Filter** training methods.

### Double Pendulum Dynamics

The system has **4 states**:
- **θ₁**: Angle of first pendulum (rad)
- **θ₂**: Angle of second pendulum (rad)
- **ω₁**: Angular velocity of first pendulum (rad/s)
- **ω₂**: Angular velocity of second pendulum (rad/s)

The double pendulum is a classic example of a **chaotic system** - small changes in initial conditions can lead to dramatically different trajectories.

### RHONN Configuration Options

**4 Configuration Levels** available (per neuron or uniform):

1. **Config 1: Minimal** (6 features) - Basic sigmoid + linear terms
2. **Config 2: Standard** (10 features) ⭐ RECOMMENDED - Interactions + quadratic terms
3. **Config 3: Extended** (15 features) - Additional nonlinear terms + angle differences
4. **Config 4: Full** (20 features) - Maximum expressiveness with cos terms

### Key Features

- ✅ **Per-neuron configuration**: Different RHONN structure for each state
- ✅ **Advanced noise types**: Gaussian, Laplacian, Student-t, Cauchy, bimodal, impulsive
- ✅ **Comprehensive visualization**: State trajectories, phase space, errors

### Usage

Run cells in order. Main configuration in **Simulation cell**:
- Set `USE_PER_NEURON_CONFIG = True/False`
- Choose `RHONN_CONFIG_ID` or `per_neuron_configs`
- Adjust noise parameters and hyperparameters


# Neural Identifier Training with Particle Filters - Double Pendulum System

In [36]:
# Neural Identifier Training with Particle Filters - Double Pendulum System

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time

# ============================================================
# 1) True nonlinear system (Double Pendulum)
# ============================================================
def plant_dynamics(x_state, u=0):
    """
    Continuous dynamics for Double Pendulum System.
    x_state = [theta1, theta2, omega1, omega2]
    where:
        theta1, theta2: angles of pendulum 1 and 2 (rad)
        omega1, omega2: angular velocities (rad/s)
    
    Equations (derived from Lagrangian mechanics):
    See: https://www.myphysicslab.com/pendulum/double-pendulum-en.html
    
    Parameters:
    - m1, m2: masses of pendulums (kg)
    - L1, L2: lengths of pendulums (m)
    - g: gravity (9.81 m/s^2)
    - b1, b2: damping coefficients
    """
    # System parameters
    m1 = 1.0   # mass of pendulum 1 (kg)
    m2 = 1.0   # mass of pendulum 2 (kg)
    L1 = 1.0   # length of pendulum 1 (m)
    L2 = 1.0   # length of pendulum 2 (m)
    g = 9.81   # gravity (m/s^2)
    b1 = 0.1   # damping coefficient 1
    b2 = 0.1   # damping coefficient 2
    
    theta1, theta2, omega1, omega2 = x_state
    
    # Differences
    delta = theta2 - theta1
    
    # Denominators for the equations
    den1 = (m1 + m2) * L1 - m2 * L1 * np.cos(delta)**2
    den2 = (L2 / L1) * den1
    
    # Derivatives of angles
    dtheta1_dt = omega1
    dtheta2_dt = omega2
    
    # Derivatives of angular velocities (from Lagrangian)
    domega1_dt = (m2 * L1 * omega1**2 * np.sin(delta) * np.cos(delta) +
                  m2 * g * np.sin(theta2) * np.cos(delta) +
                  m2 * L2 * omega2**2 * np.sin(delta) -
                  (m1 + m2) * g * np.sin(theta1) -
                  b1 * omega1) / den1
    
    domega2_dt = (-m2 * L2 * omega2**2 * np.sin(delta) * np.cos(delta) +
                  (m1 + m2) * g * np.sin(theta1) * np.cos(delta) -
                  (m1 + m2) * L1 * omega1**2 * np.sin(delta) -
                  (m1 + m2) * g * np.sin(theta2) -
                  b2 * omega2) / den2
    
    return np.array([dtheta1_dt, dtheta2_dt, domega1_dt, domega2_dt])

def plant(x_k, u_k, dt=0.01, process_noise_type='gaussian', process_noise_std=1e-2):
    """
    One Euler step of the discrete plant with ADVANCED process noise.
    """
    # Simple Euler integration
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot

    # Initialize noise container
    noise = np.zeros_like(x_kp1)

    # --- NOISE GENERATION ---
    if process_noise_type == 'laplacian':
        noise = np.random.laplace(0, process_noise_std, size=x_kp1.shape)

    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std
        noise = np.random.uniform(-a, a, size=x_kp1.shape)

    elif process_noise_type == 'student_t':
        df = 3 
        noise = np.random.standard_t(df, size=x_kp1.shape) * process_noise_std

    elif process_noise_type == 'cauchy':
        noise = np.random.standard_cauchy(size=x_kp1.shape) * process_noise_std

    elif process_noise_type == 'bimodal':
        means = np.random.choice([1, -1], size=x_kp1.shape)
        gauss_part = np.random.normal(0, process_noise_std * 0.5, size=x_kp1.shape)
        noise = (means * 2 * process_noise_std) + gauss_part

    elif process_noise_type == 'impulsive':
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)
        prob_spike = 0.05
        mask = np.random.choice([0, 1], size=x_kp1.shape, p=[1-prob_spike, prob_spike])
        spikes = np.random.normal(0, 10 * process_noise_std, size=x_kp1.shape)
        noise += mask * spikes

    else:  # gaussian (Default)
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)

    return x_kp1 + noise


In [37]:

# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-1, R_init=1e-2, P_init=1.0, eta=1.0):
        """
        Parameters:
        -----------
        num_neurons : int
            Number of neurons (one per state dimension)
        num_weights_per_neuron : int or list
            If int: same number of weights for all neurons
            If list: different number of weights per neuron (for per-neuron configs)
        """
        self.num_neurons = num_neurons
        self.eta = eta
        self.weights, self.P, self.Q, self.R = [], [], [], []
        
        # Handle both uniform and per-neuron weight dimensions
        if isinstance(num_weights_per_neuron, int):
            weight_dims = [num_weights_per_neuron] * num_neurons
        else:
            weight_dims = num_weights_per_neuron

        for i in range(num_neurons):
            n_weights = weight_dims[i]
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(n_weights) * 0.05
            self.weights.append(w_i)
            self.P.append(np.eye(n_weights) * P_init)
            self.Q.append(np.eye(n_weights) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous):
        # Series-Parallel: Use measured true state (chi_k) to build feature vector
        x_state_for_z = np.copy(chi_k) 

        for i in range(self.num_neurons):
            # Get feature vector for this specific neuron
            z_i = construct_z_vector(x_state_for_z, neuron_idx=i)
            H_i = z_i.reshape(-1, 1)
            
            n_weights = len(self.weights[i])
            P_pred = self.P[i] + self.Q[i] + np.eye(n_weights) * 1e-8
            
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-10: M_i = 1e-10

            x_hat_pred_i = self.weights[i] @ z_i
            e_i = chi_kp1[i] - x_hat_pred_i

            K_i = (P_pred @ H_i).flatten() / M_i

            eta = self.eta * (1.0 / (1.0 + np.abs(e_i) * 0.01))
            self.weights[i] += eta * K_i * e_i

            self.P[i] = (np.eye(n_weights) - np.outer(K_i, H_i.ravel())) @ P_pred
            
            # PSD enforcement
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            if np.min(np.linalg.eigvals(self.P[i])) <= 0:
                self.P[i] += np.eye(n_weights) * 1e-5


In [38]:

# ============================================================
# 2) RHONN structure - DOUBLE PENDULUM WITH CUSTOM EQUATIONS
# ============================================================
def sigmoidal(z, alpha=1.0, beta=1.0, gamma=0.0):
    """Sigmoid S(z)."""
    return alpha / (1.0 + np.exp(-beta * z)) + gamma

# ============================================================
# PREDEFINED RHONN CONFIGURATIONS
# ============================================================

def rhonn_config_1(x_est, scale=1):
    """Config 1: Minimal - 4 features"""
    theta1, theta2, omega1, omega2 = x_est
    s1 = sigmoidal(theta1 * scale)
    s2 = sigmoidal(theta2 * scale)
    s3 = sigmoidal(omega1 * scale)
    s4 = sigmoidal(omega2 * scale)
    
    return np.array([s1, s2, s3, s4])

def rhonn_config_2(x_est, scale=1.0):
    """Config 2: Standard - 8 features"""
    theta1, theta2, omega1, omega2 = x_est
    s1 = sigmoidal(theta1 * scale)
    s2 = sigmoidal(theta2 * scale)
    s3 = sigmoidal(omega1 * scale)
    s4 = sigmoidal(omega2 * scale)
    
    return np.array([s1, s2, s3, s4, s1*s2, s3*s4, s1**2, s2**2])

def rhonn_config_3(x_est, scale=1.0):
    """Config 3: Extended - 6 features"""
    theta1, theta2, omega1, omega2 = x_est
    s1 = sigmoidal(theta1 * scale)
    s2 = sigmoidal(theta2 * scale)
    s3 = sigmoidal(omega1 * scale)
    s4 = sigmoidal(omega2 * scale)
    
    return np.array([s1*s2, s3*s4, s1**2, s2**2, s3**2, s4**2])

def rhonn_config_4(x_est, scale=1.0):
    """Config 4: Full - 14 features"""
    theta1, theta2, omega1, omega2 = x_est
    s1 = sigmoidal(theta1 * scale)
    s2 = sigmoidal(theta2 * scale)
    s3 = sigmoidal(omega1 * scale)
    s4 = sigmoidal(omega2 * scale)
    
    return np.array([
        s1, s2, s3, s4,
        s1*s2, s1*s3, s1*s4, s2*s3, s2*s4, s3*s4,
        s1**2, s2**2, s3**2, s4**2
    ])


# ============================================================
# 🎯 CUSTOM EQUATIONS - Adaptadas a la física del péndulo doble
# ============================================================

def custom_theta1_equation(x_est, scale=1.0):
    """
    θ₁ (ángulo primer péndulo) - 9 features
    Balance entre simplicidad y expresividad
    """
    theta1, theta2, omega1, omega2 = x_est
    s1 = sigmoidal(theta1 * scale)
    s2 = sigmoidal(theta2 * scale)
    s3 = sigmoidal(omega1 * scale)
    delta = theta2 - theta1
    
    return np.array([
        s1, s1**2, s1*s2, s3,
        sigmoidal(delta * scale),
        np.sin(theta1), np.cos(theta1),
        omega1, 1.0
    ])

def custom_theta2_equation(x_est, scale=1.0):
    """
    θ₂ (ángulo segundo péndulo) - 13 features
    Mayor complejidad por comportamiento caótico
    """
    theta1, theta2, omega1, omega2 = x_est
    s1 = sigmoidal(theta1 * scale)
    s2 = sigmoidal(theta2 * scale)
    s4 = sigmoidal(omega2 * scale)
    delta = theta2 - theta1
    
    return np.array([
        s2, s2**2, s2**3, s1*s2, s2*s4,
        sigmoidal(delta * scale), sigmoidal(-delta * scale),
        np.sin(theta2), np.cos(theta2), np.sin(delta),
        omega2, omega1*omega2, 1.0
    ])

def custom_omega1_equation(x_est, scale=1.0):
    """
    ω₁ (velocidad angular primer péndulo) - 11 features
    Dinámica de aceleración
    """
    theta1, theta2, omega1, omega2 = x_est
    s1 = sigmoidal(theta1 * scale)
    s3 = sigmoidal(omega1 * scale)
    s4 = sigmoidal(omega2 * scale)
    delta = theta2 - theta1
    
    return np.array([
        s3, s3**2, s1*s3, s3*s4,
        sigmoidal(delta * scale), np.sin(theta1)*s3,
        omega1, omega1**2, omega1*omega2,
        np.cos(delta), 1.0
    ])

def custom_omega2_equation(x_est, scale=1.0):
    """
    ω₂ (velocidad angular segundo péndulo) - 15 features
    Máxima complejidad para capturar caos
    """
    theta1, theta2, omega1, omega2 = x_est
    s1 = sigmoidal(theta1 * scale)
    s2 = sigmoidal(theta2 * scale)
    s3 = sigmoidal(omega1 * scale)
    s4 = sigmoidal(omega2 * scale)
    delta = theta2 - theta1
    
    return np.array([
        s4, s4**2, s4**3, s2*s4, s3*s4, s1*s2*s4,
        sigmoidal(delta * scale), np.sin(theta2)*s4,
        np.cos(delta), np.sin(delta)*s4,
        omega2, omega2**2, omega1*omega2, omega1**2*omega2, 1.0
    ])


# ============================================================
# CONFIGURATION SYSTEM
# ============================================================

CUSTOM_NEURON_EQUATIONS = None

def rhonn_custom_equation(x_est, neuron_idx=0, scale=1.0):
    """Apply custom user-defined equation for specific neuron."""
    if CUSTOM_NEURON_EQUATIONS is None or neuron_idx >= len(CUSTOM_NEURON_EQUATIONS):
        raise ValueError(f"Custom equation for neuron {neuron_idx} not defined!")
    return CUSTOM_NEURON_EQUATIONS[neuron_idx](x_est, scale=scale)

def set_custom_neuron_equations(equation_functions):
    """
    Set custom equations for each neuron.
    
    Example:
    --------
    # Usar ecuaciones predefinidas del péndulo doble:
    set_custom_neuron_equations([
        custom_theta1_equation,
        custom_theta2_equation,
        custom_omega1_equation,
        custom_omega2_equation
    ])
    """
    global CUSTOM_NEURON_EQUATIONS
    CUSTOM_NEURON_EQUATIONS = equation_functions
    
    state_names = ['theta1', 'theta2', 'omega1', 'omega2']
    test_state = np.array([0.5, 0.3, 1.0, -0.5])
    
    print(f"\n{'='*70}")
    print(f"✏️  ECUACIONES PERSONALIZADAS CONFIGURADAS:")
    print(f"{'='*70}")
    for idx, func in enumerate(CUSTOM_NEURON_EQUATIONS):
        state_name = state_names[idx] if idx < len(state_names) else f'state_{idx}'
        test_output = func(test_state, scale=1.0)
        print(f"   Neuron {idx} ({state_name}): {func.__name__} - {len(test_output)} features")
    print(f"{'='*70}\n")
    
    return [len(func(test_state, scale=1.0)) for func in equation_functions]


RHONN_CONFIGS = {
    1: {'func': rhonn_config_1, 'name': 'Minimal', 'n_features': 4},
    2: {'func': rhonn_config_2, 'name': 'Standard', 'n_features': 8},
    3: {'func': rhonn_config_3, 'name': 'Extended', 'n_features': 6},
    4: {'func': rhonn_config_4, 'name': 'Full', 'n_features': 14},
    'custom': {'func': rhonn_custom_equation, 'name': 'Custom', 'n_features': None},
}

NEURON_RHONN_CONFIGS = None

def set_rhonn_config(config_id=2, num_neurons=None):
    """Set uniform configuration for all neurons"""
    global NEURON_RHONN_CONFIGS
    
    if config_id not in RHONN_CONFIGS:
        raise ValueError(f"Invalid config_id: {config_id}")
    
    NEURON_RHONN_CONFIGS = [config_id] * (num_neurons or 1)
    
    config = RHONN_CONFIGS[config_id]
    print(f"\n{'='*60}")
    print(f"🔧 RHONN: {config['name']} (ID: {config_id})")
    if config['n_features']:
        print(f"   Features: {config['n_features']}")
    if num_neurons:
        print(f"   Neuronas: {num_neurons}")
    print(f"{'='*60}\n")
    
    if config_id == 'custom':
        if CUSTOM_NEURON_EQUATIONS is None:
            raise ValueError("Llama set_custom_neuron_equations() primero!")
        test_state = np.array([0.5, 0.3, 1.0, -0.5])
        return len(CUSTOM_NEURON_EQUATIONS[0](test_state, scale=1.0))
    
    return config['n_features']

def set_rhonn_config_per_neuron(config_list):
    """Set different configuration for each neuron"""
    global NEURON_RHONN_CONFIGS
    
    for idx, config_id in enumerate(config_list):
        if config_id not in RHONN_CONFIGS:
            raise ValueError(f"Invalid config_id {config_id} at neuron {idx}")
        if config_id == 'custom' and CUSTOM_NEURON_EQUATIONS is None:
            raise ValueError(f"Neuron {idx} usa custom pero no hay ecuaciones definidas!")
    
    NEURON_RHONN_CONFIGS = list(config_list)
    
    state_names = ['theta1', 'theta2', 'omega1', 'omega2']
    test_state = np.array([0.5, 0.3, 1.0, -0.5])
    n_features_list = []
    
    print(f"\n{'='*70}")
    print(f"🔧 CONFIGURACIÓN POR NEURONA:")
    print(f"{'='*70}")
    for idx, config_id in enumerate(NEURON_RHONN_CONFIGS):
        config = RHONN_CONFIGS[config_id]
        state_name = state_names[idx] if idx < len(state_names) else f'state_{idx}'
        
        if config_id == 'custom':
            n_features = len(CUSTOM_NEURON_EQUATIONS[idx](test_state, scale=1.0))
            print(f"   {idx} ({state_name}): CUSTOM - {CUSTOM_NEURON_EQUATIONS[idx].__name__} ({n_features})")
        else:
            n_features = config['n_features']
            print(f"   {idx} ({state_name}): Config {config_id} - {config['name']} ({n_features})")
        
        n_features_list.append(n_features)
    print(f"{'='*70}\n")
    
    return n_features_list

def construct_z_vector(x_est, neuron_idx=0, scale=1.0):
    """Construct feature vector for a specific neuron"""
    if NEURON_RHONN_CONFIGS is None:
        raise ValueError("Llama set_rhonn_config() primero!")
    
    config_id = NEURON_RHONN_CONFIGS[neuron_idx if neuron_idx < len(NEURON_RHONN_CONFIGS) else -1]
    
    if config_id == 'custom':
        return rhonn_custom_equation(x_est, neuron_idx, scale)
    else:
        return RHONN_CONFIGS[config_id]['func'](x_est, scale)

def RHONN_predict(x_state_for_z, w_neuron, neuron_idx=0):
    """Predict next-state component with RHONN neuron"""
    z_i = construct_z_vector(x_state_for_z, neuron_idx=neuron_idx)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Neuron {neuron_idx}: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)


# ============================================================
# 🚀 QUICK START - Ecuaciones personalizadas
# ============================================================
print("\n" + "="*70)
print("💡 USO DE ECUACIONES PERSONALIZADAS:")
print("="*70)
print("\n# Activar ecuaciones adaptadas al péndulo doble:")
print("set_custom_neuron_equations([")
print("    custom_theta1_equation,   # 9 features")
print("    custom_theta2_equation,   # 13 features")
print("    custom_omega1_equation,   # 11 features")
print("    custom_omega2_equation    # 15 features")
print("])")
print("\nset_rhonn_config_per_neuron(['custom', 'custom', 'custom', 'custom'])")
print("="*70 + "\n")



💡 USO DE ECUACIONES PERSONALIZADAS:

# Activar ecuaciones adaptadas al péndulo doble:
set_custom_neuron_equations([
    custom_theta1_equation,   # 9 features
    custom_theta2_equation,   # 13 features
    custom_omega1_equation,   # 11 features
    custom_omega2_equation    # 15 features
])

set_rhonn_config_per_neuron(['custom', 'custom', 'custom', 'custom'])



In [39]:

# ============================================================
# 5) Simulation - Double Pendulum
# ============================================================
if __name__ == "__main__":
    # ============================================================
    # 🔧 SET RHONN CONFIGURATION HERE
    # ============================================================
    # 
    # OPTION 1: Same configuration for all neurons (UNIFORM)
    # ------------------------------------------------------
    # Choose configuration: 1, 2, 3, or 4
    # - Config 1: Minimal (Linear + Sigmoid) - 6 features
    # - Config 2: Standard (Interactions + Quadratic) - 10 features  ⭐ RECOMMENDED
    # - Config 3: Extended (More Nonlinear) - 15 features
    # - Config 4: Full (Maximum Expressiveness) - 20 features
    
    USE_PER_NEURON_CONFIG = False  # 👈 Set to True to use different configs per neuron
    
    if not USE_PER_NEURON_CONFIG:
        # UNIFORM: Same config for all neurons
        RHONN_CONFIG_ID = 2  # 👈 CHANGE THIS TO SELECT CONFIGURATION
        num_neurons = 4  # theta1, theta2, omega1, omega2
        num_weights_per_neuron = set_rhonn_config(RHONN_CONFIG_ID, num_neurons=num_neurons)
    else:
        # OPTION 2: Different configuration per neuron (PER-NEURON)
        # ----------------------------------------------------------
        # Example: Use Config 2 for all angle states, Config 3 for velocity states
        # Config IDs: [theta1_neuron, theta2_neuron, omega1_neuron, omega2_neuron]
        
        per_neuron_configs = [2, 2, 3, 3]  # 👈 CHANGE THIS FOR PER-NEURON CONFIGS
        num_weights_per_neuron = set_rhonn_config_per_neuron(per_neuron_configs)
        num_neurons = len(per_neuron_configs)
        
        # Show current assignment
        print_current_neuron_configs()
    
    # ============================================================
    
    # --- Reproducibility seed ---
    # SEED = np.random.randint(0, 10000)
    SEED = 7517
    np.random.seed(SEED)
    print(f"🎲 Semilla aleatoria (seed): {SEED}")
    print("   (Para reproducibilidad de resultados)\n")
    
    # --- Simulation settings ---
    n_steps = 1000
    dt = 0.02
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

    process_noise_type = 'laplacian' # 'gaussian', 'laplacian', 'uniform', 'student_t', 'cauchy', 'bimodal', 'impulsive'
    process_noise_std = 0.02

    measurement_noise_std = 0.1

    # Tuning parameters for 4-state double pendulum
    Q_std_per_neuron = [2.0, 3.0, 1.0, 1.0]  # theta1, theta2, omega1, omega2
    R_std_per_neuron = [0.5, 0.5, 0.5, 0.5]  # theta1, theta2, omega1, omega2

    # --- True system init ---
    x_true = np.zeros((n_steps, 4))
    x_true[0] = [np.pi/4, np.pi/6, 0.0, 0.0]  # theta1=45°, theta2=30°, both at rest
    
    num_particles = 1200

    # --- Initial weights ---
    # Handle both uniform and per-neuron weight dimensions
    if isinstance(num_weights_per_neuron, list):
        common_initial_weights = [np.random.uniform(-0.5, 0.5, nw) for nw in num_weights_per_neuron]
    else:
        common_initial_weights = [np.random.uniform(-0.5, 0.5, num_weights_per_neuron) for _ in range(num_neurons)]

    # --- Instantiate Trainers ---
    ekf_trainer = EKF_RHONN_Trainer(num_neurons, num_weights_per_neuron, initial_weights=common_initial_weights, 
                                    eta=1.0, Q_init=2.5e-1, R_init=1.0e-7, P_init=2.0)
    pf_trainer = PF_RHONN_Trainer(num_neurons, num_weights_per_neuron, n_particles=num_particles, 
                                  initial_weights=common_initial_weights, 
                                  Q_std=Q_std_per_neuron, R_std=R_std_per_neuron, ess_threshold=0.5*num_particles)


    # Initialize Particle Filter cloud
    for i in range(num_neurons):
        pf_trainer.particles[i] = np.tile(common_initial_weights[i], (pf_trainer.n_particles, 1))
        pf_trainer.particles[i] += np.random.normal(size=(pf_trainer.n_particles, pf_trainer.weight_dims[i])) * 0.1

    # Print PF configuration
    print(pf_trainer.get_info())

    # Storage
    x_hat_ekf = np.zeros((n_steps, 4))
    x_hat_ekf[0] = x_true[0]
    x_hat_pf = np.zeros((n_steps, 4))
    x_hat_pf[0] = x_true[0]

    # Timing variables
    ekf_time_total = 0.0
    pf_time_total = 0.0

    print("Starting Double Pendulum simulation...")
    print("This system is highly nonlinear and chaotic - expect rich dynamics!\n")
    
    for k in range(n_steps - 1):
        # 1) Evolve true system
        x_true[k+1] = plant(x_true[k], 0, dt, process_noise_type, process_noise_std) + np.random.normal(0, measurement_noise_std, size=4)

        # 2) Update EKF
        t0 = time.time()
        ekf_trainer.update(x_true[k+1], x_true[k], x_hat_ekf[k])
        ekf_time_total += (time.time() - t0)
        
        # EKF prediction for next state
        w_ekf = ekf_trainer.weights
        pred_ekf = np.array([RHONN_predict(x_true[k], w_ekf[i], neuron_idx=i) for i in range(num_neurons)])
        x_hat_ekf[k+1] = pred_ekf

        # 3) Update PF
        t0 = time.time()
        pf_trainer.update(x_true[k+1], x_true[k], x_hat_pf[k])
        pf_time_total += (time.time() - t0)
        
        # PF prediction
        w_pf = pf_trainer.get_estimate()
        pred_pf = np.array([RHONN_predict(x_true[k], w_pf[i], neuron_idx=i) for i in range(num_neurons)])
        x_hat_pf[k+1] = pred_pf

        if k % 200 == 0:
            print(f"Step {k}/{n_steps-1} completed")

    # --- End Simulation ---
    avg_time_ekf = ekf_time_total / (n_steps - 1)
    avg_time_pf = pf_time_total / (n_steps - 1)
    
    print("\n" + "="*70)
    print("⏱️  TIMING RESULTS:")
    print("="*70)
    print(f"EKF:")
    print(f"   Total time: {ekf_time_total:.4f} seconds")
    print(f"   Average per step: {avg_time_ekf*1000:.4f} ms")
    print(f"\nPF:")
    print(f"   Total time: {pf_time_total:.4f} seconds")
    print(f"   Average per step: {avg_time_pf*1000:.4f} ms")
    print(f"\nSpeedup: EKF is {pf_time_total/ekf_time_total:.2f}x faster than PF")
    print("="*70 + "\n")

    # --- Compute Errors ---
    err_ekf = np.linalg.norm(x_true - x_hat_ekf, axis=1)
    err_pf = np.linalg.norm(x_true - x_hat_pf, axis=1)

    # RMSE
    rmse_ekf = np.sqrt(np.mean(err_ekf**2))
    rmse_pf = np.sqrt(np.mean(err_pf**2))

    # Per-state RMSE
    rmse_ekf_per_state = np.sqrt(np.mean((x_true - x_hat_ekf)**2, axis=0))
    rmse_pf_per_state = np.sqrt(np.mean((x_true - x_hat_pf)**2, axis=0))

    print("="*70)
    print("📊 ESTIMATION ERRORS (RMSE):")
    print("="*70)
    print(f"EKF:  {rmse_ekf:.6f}")
    print(f"PF:   {rmse_pf:.6f}")
    print(f"Difference: {abs(rmse_ekf - rmse_pf):.6f}")
    if rmse_ekf < rmse_pf:
        print(f"🏆 Winner: EKF (better by {(rmse_pf/rmse_ekf - 1)*100:.2f}%)")
    else:
        print(f"🏆 Winner: PF (better by {(rmse_ekf/rmse_pf - 1)*100:.2f}%)")
    
    print("\nPer-State RMSE:")
    state_names = ['theta1', 'theta2', 'omega1', 'omega2']
    for i, name in enumerate(state_names):
        print(f"  {name}:")
        print(f"    EKF: {rmse_ekf_per_state[i]:.6f}")
        print(f"    PF:  {rmse_pf_per_state[i]:.6f}")
    print("="*70 + "\n")

    # PF Statistics
    stats_pf = pf_trainer.get_statistics()
    print("="*70)
    print("🔬 PARTICLE FILTER STATISTICS (Final):")
    print("="*70)
    for i in range(num_neurons):
        state_name = state_names[i] if i < len(state_names) else f'state_{i}'
        print(f"\nNeuron {i} ({state_name}):")
        print(f"   ESS: {stats_pf['ess'][i]:.2f} ({stats_pf['ess_ratio'][i]*100:.1f}% of {num_particles})")
        print(f"   Max weight: {stats_pf['max_weight'][i]:.6f}")
        print(f"   Min weight: {stats_pf['min_weight'][i]:.6f}")
    print("="*70 + "\n")



🔧 RHONN: Standard (ID: 2)
   Features: 8
   Neuronas: 4

🎲 Semilla aleatoria (seed): 7517
   (Para reproducibilidad de resultados)


🔬 Particle Filter Configuration:
   Particles: 1200
   Weight dimensions per neuron: [8, 8, 8, 8]

Starting Double Pendulum simulation...
This system is highly nonlinear and chaotic - expect rich dynamics!

Step 0/999 completed
Step 200/999 completed
Step 400/999 completed
Step 600/999 completed
Step 800/999 completed

⏱️  TIMING RESULTS:
EKF:
   Total time: 0.0989 seconds
   Average per step: 0.0990 ms

PF:
   Total time: 0.6178 seconds
   Average per step: 0.6184 ms

Speedup: EKF is 6.24x faster than PF

📊 ESTIMATION ERRORS (RMSE):
EKF:  0.061098
PF:   0.182662
Difference: 0.121564
🏆 Winner: EKF (better by 198.97%)

Per-State RMSE:
  theta1:
    EKF: 0.000600
    PF:  0.024891
  theta2:
    EKF: 0.001364
    PF:  0.029181
  omega1:
    EKF: 0.032717
    PF:  0.096794
  omega2:
    EKF: 0.051578
    PF:  0.150084

🔬 PARTICLE FILTER STATISTICS (Final):



In [40]:
# ============================================================
# 6) Visualization - Double Pendulum
# ============================================================

# Create comprehensive visualization
fig = make_subplots(
    rows=5, cols=2,
    subplot_titles=(
        'Angle 1 (θ₁)', 'Angle 2 (θ₂)',
        'Angular Velocity 1 (ω₁)', 'Angular Velocity 2 (ω₂)',
        'Phase Space (θ₁ vs ω₁)', 'Phase Space (θ₂ vs ω₂)',
        'Tracking Error (Norm)', 'Tracking Error per State',
        'Double Pendulum Animation Path', ''
    ),
    specs=[
        [{"type": "scatter"}, {"type": "scatter"}],
        [{"type": "scatter"}, {"type": "scatter"}],
        [{"type": "scatter"}, {"type": "scatter"}],
        [{"type": "scatter"}, {"type": "scatter"}],
        [{"type": "scatter"}, {"type": "scatter"}]
    ],
    vertical_spacing=0.08,
    horizontal_spacing=0.12,
    row_heights=[0.18, 0.18, 0.18, 0.18, 0.28]
)

state_names = ['θ₁', 'θ₂', 'ω₁', 'ω₂']
colors = {'true': 'black', 'ekf': 'blue', 'pf': 'red'}

# Row 1-2: State trajectories
for i in range(4):
    row = (i // 2) + 1
    col = (i % 2) + 1
    
    # True state
    fig.add_trace(go.Scatter(x=t_history, y=x_true[:, i], 
                            mode='lines', name=f'True {state_names[i]}',
                            line=dict(color=colors['true'], width=2),
                            showlegend=(i==0)), row=row, col=col)
    
    # EKF estimate
    fig.add_trace(go.Scatter(x=t_history, y=x_hat_ekf[:, i],
                            mode='lines', name=f'EKF {state_names[i]}',
                            line=dict(color=colors['ekf'], width=1.5, dash='dash'),
                            showlegend=(i==0)), row=row, col=col)
    
    # PF estimate
    fig.add_trace(go.Scatter(x=t_history, y=x_hat_pf[:, i],
                            mode='lines', name=f'PF {state_names[i]}',
                            line=dict(color=colors['pf'], width=1.5, dash='dot'),
                            showlegend=(i==0)), row=row, col=col)
    
    fig.update_xaxes(title_text="Time (s)", row=row, col=col)
    fig.update_yaxes(title_text=state_names[i], row=row, col=col)

# Row 3: Phase space plots
# Phase space for theta1 vs omega1
fig.add_trace(go.Scatter(x=x_true[:, 0], y=x_true[:, 2],
                        mode='lines', name='True (θ₁-ω₁)',
                        line=dict(color=colors['true'], width=2),
                        showlegend=False), row=3, col=1)
fig.add_trace(go.Scatter(x=x_hat_ekf[:, 0], y=x_hat_ekf[:, 2],
                        mode='lines', name='EKF (θ₁-ω₁)',
                        line=dict(color=colors['ekf'], width=1.5, dash='dash'),
                        showlegend=False), row=3, col=1)
fig.add_trace(go.Scatter(x=x_hat_pf[:, 0], y=x_hat_pf[:, 2],
                        mode='lines', name='PF (θ₁-ω₁)',
                        line=dict(color=colors['pf'], width=1.5, dash='dot'),
                        showlegend=False), row=3, col=1)
fig.update_xaxes(title_text="θ₁ (rad)", row=3, col=1)
fig.update_yaxes(title_text="ω₁ (rad/s)", row=3, col=1)

# Phase space for theta2 vs omega2
fig.add_trace(go.Scatter(x=x_true[:, 1], y=x_true[:, 3],
                        mode='lines', name='True (θ₂-ω₂)',
                        line=dict(color=colors['true'], width=2),
                        showlegend=False), row=3, col=2)
fig.add_trace(go.Scatter(x=x_hat_ekf[:, 1], y=x_hat_ekf[:, 3],
                        mode='lines', name='EKF (θ₂-ω₂)',
                        line=dict(color=colors['ekf'], width=1.5, dash='dash'),
                        showlegend=False), row=3, col=2)
fig.add_trace(go.Scatter(x=x_hat_pf[:, 1], y=x_hat_pf[:, 3],
                        mode='lines', name='PF (θ₂-ω₂)',
                        line=dict(color=colors['pf'], width=1.5, dash='dot'),
                        showlegend=False), row=3, col=2)
fig.update_xaxes(title_text="θ₂ (rad)", row=3, col=2)
fig.update_yaxes(title_text="ω₂ (rad/s)", row=3, col=2)

# Row 4: Error plots
# Total error (norm)
fig.add_trace(go.Scatter(x=t_history, y=err_ekf,
                        mode='lines', name='EKF Error',
                        line=dict(color=colors['ekf'], width=2),
                        showlegend=False), row=4, col=1)
fig.add_trace(go.Scatter(x=t_history, y=err_pf,
                        mode='lines', name='PF Error',
                        line=dict(color=colors['pf'], width=2),
                        showlegend=False), row=4, col=1)
fig.update_xaxes(title_text="Time (s)", row=4, col=1)
fig.update_yaxes(title_text="Error (norm)", row=4, col=1)

# Per-state errors
err_ekf_states = np.abs(x_true - x_hat_ekf)
err_pf_states = np.abs(x_true - x_hat_pf)
for i in range(4):
    fig.add_trace(go.Scatter(x=t_history, y=err_ekf_states[:, i],
                            mode='lines', name=f'EKF {state_names[i]}',
                            line=dict(width=1.5),
                            showlegend=True), row=4, col=2)
for i in range(4):
    fig.add_trace(go.Scatter(x=t_history, y=err_pf_states[:, i],
                            mode='lines', name=f'PF {state_names[i]}',
                            line=dict(width=1.5, dash='dash'),
                            showlegend=True), row=4, col=2)
fig.update_xaxes(title_text="Time (s)", row=4, col=2)
fig.update_yaxes(title_text="Absolute Error", type="log", row=4, col=2)

# Row 5: Double pendulum path visualization (cartesian coordinates)
# Convert angles to cartesian coordinates for visualization
L1 = 1.0
L2 = 1.0

# True system path
x1_true = L1 * np.sin(x_true[:, 0])
y1_true = -L1 * np.cos(x_true[:, 0])
x2_true = x1_true + L2 * np.sin(x_true[:, 1])
y2_true = y1_true - L2 * np.cos(x_true[:, 1])

# Plot path of second pendulum mass (most interesting)
fig.add_trace(go.Scatter(x=x2_true, y=y2_true,
                        mode='lines', name='True Path (mass 2)',
                        line=dict(color=colors['true'], width=2),
                        showlegend=False), row=5, col=1)

# Add start and end points
fig.add_trace(go.Scatter(x=[x2_true[0]], y=[y2_true[0]],
                        mode='markers', name='Start',
                        marker=dict(color='green', size=10, symbol='star'),
                        showlegend=False), row=5, col=1)
fig.add_trace(go.Scatter(x=[x2_true[-1]], y=[y2_true[-1]],
                        mode='markers', name='End',
                        marker=dict(color='red', size=10, symbol='x'),
                        showlegend=False), row=5, col=1)

fig.update_xaxes(title_text="x (m)", row=5, col=1)
fig.update_yaxes(title_text="y (m)", row=5, col=1)
fig.update_xaxes(scaleanchor="y", scaleratio=1, row=5, col=1)

# Update layout
fig.update_layout(
    height=1800,
    title_text=f"Double Pendulum RHONN Identification: EKF vs Particle Filter<br>" +
               f"<sub>RMSE - EKF: {rmse_ekf:.6f}, PF: {rmse_pf:.6f} | " +
               f"Particles: {num_particles} | Process Noise: {process_noise_type}</sub>",
    showlegend=True,
    template="plotly_white",
    font=dict(size=10)
)

fig.show()

print("\n✅ Visualization complete!")
print(f"\n📈 Summary Statistics:")
print(f"   System: Double Pendulum (4 states)")
print(f"   Time steps: {n_steps}")
print(f"   Sampling time: {dt} s")
print(f"   Total duration: {n_steps*dt:.2f} s")
print(f"   Process noise: {process_noise_type}")
print(f"   Measurement noise std: {measurement_noise_std}")



✅ Visualization complete!

📈 Summary Statistics:
   System: Double Pendulum (4 states)
   Time steps: 1000
   Sampling time: 0.02 s
   Total duration: 20.00 s
   Process noise: laplacian
   Measurement noise std: 0.1
